# Explore a run

Pick a saved **run** (gives the trained model) and, independently, a **dataset** to evaluate it on. `load_predictor` builds a generic `pato.inference.Predictor` for the run — every pipeline goes through the same code path, with the train→infer asymmetry hidden inside each LightningModule's `to_inference_model()`. The notebook then loads the chosen dataset's `val` split, runs inference, and shows source / ground-truth / prediction side by side.

The dataset is always a **normalized full-image dataset** (full image + mask, no tiles, no preprocessing cache) — e.g. `nmsc-2x`. The `Predictor` tiles internally via sliding-window inference, so no per-pipeline cache needs to be on disk. This means a run trained elsewhere (e.g. on a remote GPU box, with its tile cache never synced back) still works here.

Works for `unet`, `sam`, and `nnunet` (the SAM pipeline is unified — one model class either way), and any future pipeline whose `LightningModule` implements `to_inference_model() → nn.Module`.

Runs live one-folder-each under `runs/` — Hydra's per-job dir, holding `config.yaml`, `checkpoints/`, `.hydra/`, `train.log`, `wandb/`. `list_runs(paths.runs)` returns them oldest-first; the next cell defaults to the most recent run that has a saved checkpoint (an interrupted run leaves `checkpoints/` empty).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from config import paths
from pato.dataset import DatasetViewer
from pato.experiments import (
    list_datasets,
    list_runs,
    load_predictor,
    load_run,
)
from pato.visualize import load_image, show_overlays, show_side_by_side

## 1. Pick a run

In [ ]:
list_runs(paths.runs)

In [ ]:
# `list_runs` is oldest→newest; an interrupted run leaves an empty
# `checkpoints/`, so default to the most recent run that actually has one.
completed = [n for n in list_runs(paths.runs) if load_run(paths.runs / n).best_checkpoint()]
RUN_NAME = completed[-1]   # ← or paste any name from the list above
run_path = paths.runs / RUN_NAME
run = load_run(run_path)

# Pull out the bits the notebook reuses below. Current configs store the
# train cache under `dataset.train`; older runs used `dataset.dataset_root`.
pipeline_name = run.config["pipeline"]["name"]
ds = run.config["dataset"]
dataset_root = ds.get("train", ds.get("dataset_root", "(unknown)"))

print(f"Run        : {run.name}")
print(f"Pipeline   : {pipeline_name}")
print(f"Best ckpt  : {run.best_checkpoint().name if run.best_checkpoint() else '(none)'}")
print(f"Dataset    : {dataset_root}")
print(f"Net        : {run.config['net']['_target_']}")
print(f"LR         : {run.config['lr']['learning_rate']}")
run.config

## 2. Load the predictor

One generic `Predictor` regardless of pipeline. Internally: dispatches on `config.pipeline` to load the right LightningModule, calls its `to_inference_model()`, and resolves the sliding-window tile size + overlap — from the training cache's metadata if that cache is still on disk, otherwise from the net's intrinsic input size (SAM → 1024, UNet → 512). Pass `target_size=` / `overlap=` to `load_predictor` to override.

In [ ]:
predictor = load_predictor(run_path)
print(f"pipeline    : {pipeline_name}")
print(f"target_size : {predictor.target_size}")
print(f"overlap     : {predictor.overlap}")
print(f"device      : {predictor.device}")

## 3. Pick a dataset and predict on its `val` split

The dataset is chosen **independently of the run** — always a normalized full-image dataset (full image + mask, no tiles). `list_datasets(paths.data_processed)` shows what's available (pipeline tile / feature caches are filtered out); set `DATASET` to one of those names.

The `Predictor` tiles each full slide internally via sliding-window inference, so nothing here depends on a preprocessed cache. Swap `DATASET` to sanity-check generalization across resolutions (e.g. `nmsc-5x`).

In [ ]:
list_datasets(paths.data_processed)   # normalized full-image datasets available

In [ ]:
DATASET = "nmsc-2x"   # ← pick any name from the list above

val = DatasetViewer(root=paths.data_processed / DATASET, split="val")
print(f"dataset   : {DATASET}")
print(f"val split : {len(val)} samples (full images)")

In [ ]:
sample_indices = [0]
for idx in sample_indices:
    sample = val[idx]
    predicted = predictor.predict(sample)
    print(f"--- {val.sample_ids[idx]} — image {sample.image.shape[:2]} ---")
    fig = show_overlays(
        sample.image,
        sample.mask,
        predicted,
        titles=["ground truth", f"{pipeline_name} prediction"],
        opacity=0.5,
        mask_zmax=1,
    )
    fig.show()

## 4. Predict on a bare image

`predictor.predict` takes a file path, a numpy array, or a `PatoImage` — no ground truth needed. Set `IMAGE_PATH` to a real RGB image file (e.g. a raw `.tif` slide) to try your own; left as `None` it falls back to a `val` image so the cell runs even when no raw data is on disk.

In [ ]:
# Set IMAGE_PATH to any RGB image file to predict on it, e.g.
#   IMAGE_PATH = paths.nmsc_5x / "Images" / "BCC_1.tif"
# Left as None, the notebook reuses a val image as a bare array so this
# cell runs without raw data on disk.
IMAGE_PATH = None

image = load_image(IMAGE_PATH) if IMAGE_PATH else val[1].image
predicted = predictor.predict(image)
print(f"image: {image.shape}, prediction: {predicted.shape} {predicted.dtype}")

show_overlays(
    image,
    predicted,
    titles=[f"{pipeline_name} prediction"],
    opacity=0.5,
    mask_zmax=1,
)